<a href="https://colab.research.google.com/github/everestso/everestso/blob/main/LLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# pips

!pip install aisuite
!pip -q install "aisuite[groq]"
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: docstring-parser
    Found existing installation: docstring_parser 0.17.0
    Uninstalling docstring_parser-0.17.0:
      Successfully uninstalled docstring_parser-0.17.0
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.55.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Keys
import os
import json
from google.colab import userdata
import anthropic

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

In [3]:
# LLM as a Client

import aisuite as ai

# Create an instance of the AISuite client
client = ai.Client()

In [4]:
# Message structure
prompt = "Where is Fresno State University?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

In [5]:
response = client.chat.completions.create(
    model="groq:llama-3.3-70b-versatile",
    messages=messages
)

# See the LLM response
print(response.choices[0].message.content)

Fresno State University, also known as California State University, Fresno (CSUF), is located in Fresno, California, United States. Specifically, the university's main campus is situated at 5241 N Maple Ave, Fresno, CA 93740, in the San Joaquin Valley region of California.


In [6]:
# zero-shot

prompt = (
    "Where is Fresno State University?\n\n"
    "Return the answer strictly as a JSON object with the following fields:\n"
    "- name\n"
    "- street_address\n"
    "- city\n"
    "- state\n"
    "- postal_code\n"
    "- country\n\n"
    "Do not include any explanatory text. Output JSON only."
)

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]


In [7]:
response = client.chat.completions.create(
    model="groq:llama-3.3-70b-versatile",
    messages=messages
)

# See the LLM response
print(response.choices[0].message.content)

{
  "name": "Fresno State University",
  "street_address": "5241 N Maple Ave",
  "city": "Fresno",
  "state": "California",
  "postal_code": "93740",
  "country": "United States"
}


In [8]:
import json

raw = response.choices[0].message.content

try:
    data = json.loads(raw)
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print("Model did not return valid JSON:")
    print(raw)


{
  "name": "Fresno State University",
  "street_address": "5241 N Maple Ave",
  "city": "Fresno",
  "state": "California",
  "postal_code": "93740",
  "country": "United States"
}


In [9]:
# zero-shot
location = "Elbow Room Fresno"

prompt = (
    f"Where is {location}?\n\n"
    "Return the answer strictly as a JSON object with the following fields:\n"
    "- name (official business name)\n"
    "- street_address\n"
    "- city\n"
    "- state\n"
    "- postal_code\n"
    "- country\n\n"
    "If you are uncertain about any field, use null instead of guessing.\n"
    "Do not include any explanatory text. Output JSON only."
)

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

response = client.chat.completions.create(
    model="groq:llama-3.3-70b-versatile",
    messages=messages
)

# See the LLM response
print(response.choices[0].message.content)


{
  "name": "Elbow Room",
  "street_address": "7730 N Fresno St",
  "city": "Fresno",
  "state": "California",
  "postal_code": "93650",
  "country": "United States"
}


# Using a tool

In [10]:
from typing import Dict

def get_current_local_datetime(
    city: str,
    state: str,
    country: str
) -> Dict[str, str]:
    """
    Returns the current local date and time for a given city.

    Use this tool when you need the current day of the week or
    the current local time to determine whether a business is open.

    Args:
        city (str): City where the business is located
        state (str): State or region where the business is located
        country (str): Country where the business is located

    Returns:
        dict:
            datetime_iso (str): ISO-8601 local datetime with timezone offset
            day_of_week (str): Full weekday name (e.g., "Sunday")
            timezone (str): IANA timezone identifier
    """
    output = [{
        "datetime_iso": "2026-01-04T18:42:00-08:00",
        "day_of_week": "Sunday",
        "timezone": "America/Los_Angeles"
    }, {
        "datetime_iso": "2026-01-04T00:42:00-08:00",
        "day_of_week": "Sunday",
        "timezone": "America/Los_Angeles"
    }]
    return output[1]

In [11]:
import json
import re
import aisuite as ai
from typing import TypedDict, Optional

_llm_client = ai.Client()

class BusinessHours(TypedDict):
    monday: Optional[str]
    tuesday: Optional[str]
    wednesday: Optional[str]
    thursday: Optional[str]
    friday: Optional[str]
    saturday: Optional[str]
    sunday: Optional[str]

def _extract_json_object(text: str) -> str:
    """Extract the first JSON object from a string (handles ```json fences)."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text).strip()

    if text.startswith("{") and text.endswith("}"):
        return text

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        return m.group(0)

    return text  # will fail json.loads with a useful error

def get_business_hours_llm(name: str, city: str) -> BusinessHours:
    """
    Returns weekly opening hours for a business (Monday–Sunday) as JSON.
    Inputs: name, city. Returns nulls if unknown.
    """
    prompt = f"""
You are helping populate weekly business hours.

Business name: {name}
City: {city}

Return ONLY a JSON object with keys monday..sunday.
Each value should be either:
- a string like "11:00 AM–9:00 PM" (use an en dash),
- the string "Closed",
- or null if truly unknown.

JSON schema:
{{
  "monday": string|null,
  "tuesday": string|null,
  "wednesday": string|null,
  "thursday": string|null,
  "friday": string|null,
  "saturday": string|null,
  "sunday": string|null
}}

Output JSON only (no prose, no markdown).
""".strip()

    resp = _llm_client.chat.completions.create(
        model="groq:llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_turns=0
    )

    raw = resp.choices[0].message.content
    print(f"{raw=}")
    json_text = _extract_json_object(raw)

    try:
        hours = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(
            "get_business_hours_llm: model did not return valid JSON.\n"
            f"Raw content:\n{raw}"
        ) from e

    expected = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
    return {k: hours.get(k, None) for k in expected}



In [12]:
tools = [get_current_local_datetime, get_business_hours_llm]

tool_registry = {
    "get_current_local_datetime": get_current_local_datetime,
    "get_business_hours_llm": get_business_hours_llm,
}


In [13]:
# zero-shot
location = "Elbow Room Fresno"

prompt = (
    f"You are determining whether the following business is open right now: {location}.\n\n"
    "You will be provided with the current local date and time via a tool call.\n"
    "Use that information and the business hours to determine open_now.\n\n"
    "Return the answer strictly as a JSON object with the following fields:\n"
    "- name (official business name)\n"
    "- open_now (boolean: true, false, or null)\n"
    "- hours (object with keys monday–sunday, values as strings or null)\n"
    "- street_address\n"
    "- city\n"
    "- state\n"
    "- postal_code\n"
    "- country\n\n"
    "If you are uncertain about any field, use null instead of guessing.\n"
    "Do not include any explanatory text. Output JSON only."
)


response = client.chat.completions.create(
    model="anthropic:claude-3-5-sonnet-20240620",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=0       # disables tool execution loop
)

print (f"{response=}")
# See the LLM response
print(response.choices[0].message.content)
choice = response.choices[0]
print(f"{choice.finish_reason=}")


UnboundLocalError: cannot access local variable 'response' where it is not associated with a value

# Tools Example 1

In [14]:
tools = [get_current_local_datetime, get_business_hours_llm]

tool_registry = {
    "get_current_local_datetime": get_current_local_datetime,
    "get_business_hours_llm": get_business_hours_llm,
}


In [15]:
location = "Elbow Room Fresno"

prompt = f"""
Determine whether "{location}" is open right now.
""".strip()

messages = [{"role": "user", "content": prompt}]

response = client.chat.completions.create(
    model="groq:llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=0      # disables tool execution loop
)

print (f"{response=}")
# See the LLM response
print(response.choices[0].message.content)
choice = response.choices[0]
print(f"{choice.finish_reason=}")



UnboundLocalError: cannot access local variable 'response' where it is not associated with a value

In [16]:
### TOOLS

location = "Elbow Room Fresno"

prompt = f"""
Determine whether "{location}" is open right now.
""".strip()

messages = [{"role": "user", "content": prompt}]

tools = [get_current_local_datetime, get_business_hours_llm]

response = client.chat.completions.create(
    model="anthropic:claude-sonnet-4-5-20250929",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=3       # tool execution loop limit
)

print (f"{response=}")
# See the LLM response
print(response.choices[0].message.content)
choice = response.choices[0]
print(f"{choice.finish_reason=}")



raw='{\n  "monday": "11:00 AM–10:00 PM",\n  "tuesday": "11:00 AM–10:00 PM",\n  "wednesday": "11:00 AM–10:00 PM",\n  "thursday": "11:00 AM–10:00 PM",\n  "friday": "11:00 AM–11:00 PM",\n  "saturday": "10:00 AM–11:00 PM",\n  "sunday": "10:00 AM–9:00 PM"\n}'
response=<aisuite.framework.chat_completion_response.ChatCompletionResponse object at 0x7c1602089b50>
Based on the information I gathered:

**Elbow Room Fresno is currently CLOSED.**

- **Current time in Fresno:** 12:42 AM (Sunday, January 4, 2026)
- **Sunday hours:** 10:00 AM – 9:00 PM

The restaurant closed at 9:00 PM on Sunday and will reopen at 10:00 AM later today (Sunday morning).
choice.finish_reason='stop'


In [17]:
### MISSING TOOLS

location = "Elbow Room Fresno"

prompt = f"""
Determine whether "{location}" is open right now.
""".strip()

messages = [{"role": "user", "content": prompt}]

tools = [get_current_local_datetime]

response = client.chat.completions.create(
    model="anthropic:claude-sonnet-4-5-20250929",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=3       # tool execution loop limit
)

print (f"{response=}")
# See the LLM response
print(response.choices[0].message.content)
choice = response.choices[0]
print(f"{choice.finish_reason=}")



response=<aisuite.framework.chat_completion_response.ChatCompletionResponse object at 0x7c160208a330>
The current time in Fresno, California is **12:42 AM (just after midnight) on Sunday, January 4, 2026**.

Unfortunately, I don't have access to Elbow Room Fresno's specific business hours in my available tools. However, I can tell you that at 12:42 AM on a Sunday morning, most restaurants and bars are typically closed, unless they're a late-night establishment or bar that stays open until 2 AM.

To confirm whether Elbow Room Fresno is currently open, I recommend:
1. Checking their website or social media pages for hours
2. Calling them directly
3. Looking them up on Google Maps or Yelp, which usually display current hours and whether they're open now

Based on the late hour (12:42 AM), it's more likely that they are **closed** right now, but you'll need to verify their actual operating hours to be certain.
choice.finish_reason='stop'


# Exploration Code

In [ ]:
tool_call = choice.message.tool_calls[0]

print(tool_call)


In [ ]:

tool_name = tool_call.function.name
tool_args = tool_call.function.arguments

# Tool arguments arrive as a JSON string → parse to dict
tool_args = json.loads(tool_args) if isinstance(tool_args, str) else tool_args

print(f"{tool_name=}")
print(f"{tool_args=}")

if tool_name == "get_current_local_datetime":
    tool_result = get_current_local_datetime(**tool_args)

elif tool_name == "get_business_hours_llm":
    tool_result = get_business_hours_llm(**tool_args)

else:
    raise ValueError(f"Unknown tool requested: {tool_name}")

print(f"{tool_result=}")


In [ ]:
messages.append(choice.message)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
})





In [ ]:
print(f"{tools=}")

In [ ]:
final_response = client.chat.completions.create(
    model="anthropic:claude-sonnet-4-5-20250929",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=1      # disables tool execution loop
)

print(final_response.choices[0].message.content)

In [ ]:
print(f"{final_response=}")

In [ ]:

tool_name = tool_call.function.name
tool_args = tool_call.function.arguments

# Tool arguments arrive as a JSON string → parse to dict
tool_args = json.loads(tool_args) if isinstance(tool_args, str) else tool_args

print(f"{tool_name=}")
print(f"{tool_args=}")

if tool_name == "get_current_local_datetime":
    tool_result = get_current_local_datetime(**tool_args)

elif tool_name == "get_business_hours_llm":
    tool_result = get_business_hours_llm(**tool_args)

else:
    raise ValueError(f"Unknown tool requested: {tool_name}")

print(f"{tool_result=}")


In [ ]:
messages.append(final_response)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
})





In [ ]:
print(f"{messages=}")

In [ ]:
final_response = client.chat.completions.create(
    model="anthropic:claude-sonnet-4-5-20250929",
    messages=messages,
    tools=tools,      # declared for consistency
    max_turns=1      # disables tool execution loop
)

print(final_response.choices[0].message.content)